<a href="https://colab.research.google.com/github/postnicov/ResazurinResorufin/blob/main/Image_color_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title Install and import required packages
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    'Pillow': 'PIL',
    'python-docx': 'docx',
    'colour-science': 'colour',
    'pandas': 'pandas',
}

missing_packages = [
    package for package, module in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    print('Installing:', ', '.join(missing_packages))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing_packages])
else:
    print('All required packages are already installed.')

from pathlib import Path
from io import BytesIO
from datetime import datetime
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve
from zipfile import ZipFile
import os
import re
import warnings

import colour
import numpy as np
import pandas as pd
from colour.notation import xyY_to_munsell_colour
from colour.utilities import ColourUsageWarning
from docx import Document
from docx.enum.section import WD_ORIENT
from docx.enum.table import WD_ALIGN_VERTICAL
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Inches, Pt
from PIL import Image, ImageOps
from google.colab import files

warnings.filterwarnings('ignore', category=ColourUsageWarning)


Installing: python-docx, colour-science


In [2]:
#@title Set archive and report options
DEFAULT_ZIP_URL = 'https://github.com/postnicov/ResazurinResorufin/raw/refs/heads/main/Data/GOST.zip'
REPORT_TITLE = 'Average Colour Analysis'
SUPPORTED_IMAGE_EXTENSIONS = {'.bmp', '.gif', '.jpeg', '.jpg', '.png', '.tif', '.tiff', '.webp'}

print('Default archive:', DEFAULT_ZIP_URL)
print('Supported image formats:', ', '.join(sorted(SUPPORTED_IMAGE_EXTENSIONS)))


Default archive: https://github.com/postnicov/ResazurinResorufin/raw/refs/heads/main/Data/GOST.zip
Supported image formats: .bmp, .gif, .jpeg, .jpg, .png, .tif, .tiff, .webp


In [3]:
#@title Define the image analysis and DOCX report functions
D65_XY = colour.CCS_ILLUMINANTS['CIE 1931 2 Degree Standard Observer']['D65']
C_XY = colour.CCS_ILLUMINANTS['CIE 1931 2 Degree Standard Observer']['C']
D65_WHITE_XYZ = colour.xy_to_XYZ(D65_XY)
C_WHITE_XYZ = colour.xy_to_XYZ(C_XY)


def safe_extract_zip(zip_path, destination):
    """Extract a ZIP archive while rejecting paths outside its destination."""
    destination = Path(destination).resolve()
    with ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f'Unsafe ZIP member path: {member.filename}')
        archive.extractall(destination)


def average_rgb(image_path):
    """Return the alpha-safe, integer RGB mean of an image's first frame."""
    with Image.open(image_path) as image:
        image = ImageOps.exif_transpose(image)
        rgb_image = image.convert('RGBA')
        pixels = np.asarray(rgb_image, dtype=np.float64).reshape(-1, 4)

    alpha = pixels[:, 3] / 255.0
    if np.any(alpha > 0):
        mean_rgb = np.sum(pixels[:, :3] * alpha[:, None], axis=0) / np.sum(alpha)
    else:
        mean_rgb = np.zeros(3)
    return tuple(np.rint(mean_rgb).astype(int))


def rgb_to_lab_and_munsell(rgb):
    """Convert 8-bit sRGB to CIE L*a*b* (D65) and a Munsell notation.

    The Munsell renotation data are defined for Illuminant C.  For that conversion,
    the colour is chromatically adapted from sRGB/D65 to Illuminant C.  Extremely
    saturated colours outside the published renotation gamut are minimally
    desaturated until the closest valid notation can be returned.
    """
    rgb_normalised = np.asarray(rgb, dtype=float) / 255.0
    xyz_d65 = colour.sRGB_to_XYZ(rgb_normalised)
    lab = colour.XYZ_to_Lab(xyz_d65, D65_XY)

    # Neutral colours have no hue or chroma in Munsell notation.
    if np.max(np.abs(lab[1:])) < 1.5:
        return tuple(np.round(lab, 2)), f'N {lab[0] / 10:.1f}/'

    # The published renotation lattice has chromatic values from 1 to 9.
    # Preserve the original Lab result while using the nearest supported value
    # only when a colour lies outside that lattice.
    munsell_lab = np.array([np.clip(lab[0], 10.0, 90.0), lab[1], lab[2]])
    lightness_was_clipped = not np.isclose(munsell_lab[0], lab[0])

    for chroma_scale in np.linspace(1.0, 0.0, 101):
        candidate_lab = np.array([
            munsell_lab[0],
            munsell_lab[1] * chroma_scale,
            munsell_lab[2] * chroma_scale,
        ])
        candidate_xyz_d65 = colour.Lab_to_XYZ(candidate_lab, D65_XY)
        candidate_xyz_c = colour.adaptation.chromatic_adaptation(
            candidate_xyz_d65, D65_WHITE_XYZ, C_WHITE_XYZ
        )
        try:
            munsell = xyY_to_munsell_colour(colour.XYZ_to_xyY(candidate_xyz_c))
            if lightness_was_clipped or chroma_scale < 1.0:
                munsell += ' (nearest valid)'
            return tuple(np.round(lab, 2)), munsell
        except Exception:
            continue

    return tuple(np.round(lab, 2)), 'Unavailable'


def round_to_half(value):
    """Round a positive decimal value to the nearest 0.5 using half-up rounding."""
    return np.floor(float(value) * 2.0 + 0.5) / 2.0


def format_half_step(value):
    """Format a half-step number without a trailing '.0' for whole numbers."""
    rounded_value = round_to_half(value)
    if np.isclose(rounded_value, round(rounded_value)):
        return str(int(round(rounded_value)))
    return f'{rounded_value:.1f}'


def munsell_to_half_step(munsell_code):
    """Round the hue, value, and chroma components of a Munsell code to 0.5."""
    nearest_suffix = ' (nearest valid)'
    suffix = nearest_suffix if munsell_code.endswith(nearest_suffix) else ''
    core = munsell_code.removesuffix(nearest_suffix)

    neutral_match = re.fullmatch(r'N\s+(\d+(?:\.\d+)?)/', core)
    if neutral_match:
        return f'N {format_half_step(neutral_match.group(1))}/' + suffix

    chromatic_match = re.fullmatch(
        r'(\d+(?:\.\d+)?)([A-Z]+)\s+(\d+(?:\.\d+)?)/(\d+(?:\.\d+)?)', core
    )
    if not chromatic_match:
        return munsell_code

    hue, hue_letter, value, chroma = chromatic_match.groups()
    return (
        f'{format_half_step(hue)}{hue_letter} '
        f'{format_half_step(value)}/{format_half_step(chroma)}'
        f'{suffix}'
    )


def roman_to_integer(roman):
    """Convert a valid Roman numeral string to an integer for natural ID ordering."""
    values = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}
    total = 0
    previous = 0
    for character in reversed(roman):
        value = values[character]
        total += -value if value < previous else value
        previous = max(previous, value)
    return total


def identifier_sort_key(row):
    """Order IDs as I, II, III, Ia, IIa, IIIa, then other file names."""
    identifier = row['ID']
    match = re.fullmatch(r'([IVXLCDM]+)([a-z]*)', identifier, flags=re.IGNORECASE)
    if match:
        roman, suffix = match.groups()
        return (0, suffix.casefold(), roman_to_integer(roman.upper()), identifier.casefold())
    return (1, identifier.casefold(), 0, identifier.casefold())


def make_swatch(rgb):
    """Return a rectangular PNG colour swatch in memory."""
    swatch = Image.new('RGB', (420, 140), color=tuple(rgb))
    output = BytesIO()
    swatch.save(output, format='PNG')
    output.seek(0)
    return output


def build_docx(rows, output_path, title=REPORT_TITLE):
    """Create a landscape DOCX report with RGB and CIE Lab channels in individual columns."""
    document = Document()
    section = document.sections[0]
    section.orientation = WD_ORIENT.LANDSCAPE
    section.page_width, section.page_height = section.page_height, section.page_width
    section.top_margin = Inches(0.5)
    section.bottom_margin = Inches(0.5)
    section.left_margin = Inches(0.5)
    section.right_margin = Inches(0.5)

    heading = document.add_heading(title, level=0)
    heading.alignment = WD_ALIGN_PARAGRAPH.CENTER
    note = document.add_paragraph(
        'Mean RGB values are calculated from all image pixels. CIE L*a*b* values use '
        'the CIE 1931 2-degree observer and D65 illuminant. Munsell notation is '
        'calculated from chromatically adapted Illuminant C coordinates. The final '
        'Munsell column rounds hue, value, and chroma to the nearest 0.5.'
    )
    note.alignment = WD_ALIGN_PARAGRAPH.CENTER
    for run in note.runs:
        run.font.size = Pt(8)

    table = document.add_table(rows=1, cols=10)
    table.style = 'Table Grid'
    table.autofit = False
    headers = ['ID', 'R', 'G', 'B', 'L*', 'a*', 'b*', 'Munsell code', 'Munsell code\n(0.5)', 'Average RGB colour']
    widths = [
        Inches(0.45), Inches(0.45), Inches(0.45), Inches(0.45),
        Inches(0.65), Inches(0.65), Inches(0.65), Inches(1.25),
        Inches(1.5), Inches(2.15),
    ]
    for index, (header, width) in enumerate(zip(headers, widths)):
        cell = table.rows[0].cells[index]
        cell.width = width
        paragraph = cell.paragraphs[0]
        paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
        run = paragraph.add_run(header)
        run.bold = True

    for row in rows:
        cells = table.add_row().cells
        for index, width in enumerate(widths):
            cells[index].width = width
            cells[index].vertical_alignment = WD_ALIGN_VERTICAL.CENTER

        cells[0].text = row['ID']
        cells[1].text = str(row['RGB'][0])
        cells[2].text = str(row['RGB'][1])
        cells[3].text = str(row['RGB'][2])
        cells[4].text = '{:.2f}'.format(row['Lab'][0])
        cells[5].text = '{:.2f}'.format(row['Lab'][1])
        cells[6].text = '{:.2f}'.format(row['Lab'][2])
        cells[7].text = row['Munsell code']
        cells[8].text = row['Munsell code (0.5)']
        swatch_paragraph = cells[9].paragraphs[0]
        swatch_paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
        swatch_paragraph.add_run().add_picture(make_swatch(row['RGB']), width=Inches(2.0))

    document.save(output_path)


def analyse_zip(zip_path, source_name):
    """Analyse every supported image in a ZIP archive and return its report path and table."""
    with TemporaryDirectory() as temporary_directory:
        extraction_directory = Path(temporary_directory) / 'images'
        extraction_directory.mkdir()
        safe_extract_zip(zip_path, extraction_directory)
        image_paths = sorted(
            path for path in extraction_directory.rglob('*')
            if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
        )
        if not image_paths:
            raise ValueError('No supported image files were found in the ZIP archive.')

        rows = []
        failed_files = []
        for image_path in image_paths:
            try:
                rgb = average_rgb(image_path)
                lab, munsell = rgb_to_lab_and_munsell(rgb)
                rows.append({
                    'ID': image_path.stem,
                    'RGB': rgb,
                    'Lab': lab,
                    'Munsell code': munsell,
                    'Munsell code (0.5)': munsell_to_half_step(munsell),
                })
            except Exception as error:
                failed_files.append(f'{image_path.name}: {error}')

    if not rows:
        raise RuntimeError('No images could be analysed.\n' + '\n'.join(failed_files))

    rows.sort(key=identifier_sort_key)

    safe_name = re.sub(r'[^A-Za-z0-9_-]+', '_', source_name).strip('_') or 'images'
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_path = Path('/content') / f'{safe_name}_colour_analysis_{timestamp}.docx'
    build_docx(rows, output_path, title=f'{REPORT_TITLE}: {source_name}')

    preview = pd.DataFrame([
        {
            'ID': row['ID'],
            'R': row['RGB'][0],
            'G': row['RGB'][1],
            'B': row['RGB'][2],
            'L*': row['Lab'][0],
            'a*': row['Lab'][1],
            'b*': row['Lab'][2],
            'Munsell code': row['Munsell code'],
            'Munsell code (0.5)': row['Munsell code (0.5)'],
        }
        for row in rows
    ])
    if failed_files:
        print('Files skipped because they could not be read:')
        print('\n'.join(failed_files))
    return output_path, preview


In [4]:
#@title Download the GOST archive and immediately download its DOCX report
downloaded_zip = Path('/content/GOST.zip')
print('Downloading the archive...')
urlretrieve(DEFAULT_ZIP_URL, downloaded_zip)

reference_report_path, results_preview = analyse_zip(downloaded_zip, 'GOST')
print(f'Created reference report: {reference_report_path.name}')
# This browser-download command runs now; the later manual-upload cell is independent.
files.download(str(reference_report_path))
display(results_preview)


Created reference report: GOST_colour_analysis_20260805_114834.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,ID,R,G,B,L*,a*,b*,Munsell code,Munsell code (0.5)
0,I,193,158,233,70.65,27.25,-32.97,3.5P 7.0/10.1,3.5P 7/10
1,II,247,152,187,73.43,39.99,-2.90,4.9RP 7.3/9.6,5RP 7.5/9.5
2,III,255,213,227,89.11,16.86,-1.37,4.4RP 8.9/4.4,4.5RP 9/4.5
3,Ia,240,218,222,88.90,8.24,0.80,7.1RP 8.9/2.5,7RP 9/2.5
4,IIa,255,128,139,68.43,49.21,16.84,2.1R 6.8/11.5,2R 7/11.5
5,IIIa,255,255,240,99.64,-2.54,7.17,1.1GY 9.0/0.8 (nearest valid),1GY 9/1 (nearest valid)


In [5]:
#@title Optional: upload another ZIP archive and immediately download its DOCX report
RUN_MANUAL_UPLOAD = False #@param {type:"boolean"}

if not RUN_MANUAL_UPLOAD:
    print('Manual upload is skipped. Set RUN_MANUAL_UPLOAD to True only when you want to analyse another ZIP file.')
else:
    uploaded_files = files.upload()
    if not uploaded_files:
        print('No file was uploaded.')
    else:
        uploaded_name, uploaded_bytes = next(iter(uploaded_files.items()))
        uploaded_zip = Path('/content') / Path(uploaded_name).name
        uploaded_zip.write_bytes(uploaded_bytes)

        report_path, results_preview = analyse_zip(uploaded_zip, uploaded_zip.stem)
        print(f'Created report: {report_path.name}')
        files.download(str(report_path))  # Starts an immediate download in the browser.
        display(results_preview)


Manual upload is skipped. Set RUN_MANUAL_UPLOAD to True only when you want to analyse another ZIP file.
